In [2]:
import subprocess
subprocess.run(["pip", "install", "pytorch-tabnet", "boruta"], capture_output=True)

import pandas as pd
import numpy as np
import sqlite3
import joblib
import os
import warnings
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

from lightgbm import LGBMClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, KFold
from sklearn.metrics import (roc_auc_score, classification_report,
                              f1_score, brier_score_loss)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import RFECV
from boruta import BorutaPy
from pytorch_tabnet.tab_model import TabNetClassifier
import shap
import matplotlib.pyplot as plt

os.makedirs("../output", exist_ok=True)
print("All imports successful!")

All imports successful!


In [3]:
conn = sqlite3.connect("../data/attrition.db")
df = pd.read_sql("SELECT * FROM employees", conn)
conn.close()

label_cols = ["BusinessTravel", "Department", "EducationField",
              "Gender", "JobRole", "MaritalStatus", "OverTime"]
le = LabelEncoder()
df_model = df.copy()
for col in label_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

role_avg = df_model.groupby("JobRole")["MonthlyIncome"].transform("mean")
df_model["SalaryToRoleRatio"]     = (df_model["MonthlyIncome"] / role_avg).round(3)
df_model["TenureStabilityScore"]  = (df_model["YearsAtCompany"] /
                                      (df_model["TotalWorkingYears"] + 1)).round(3)
df_model["SatisfactionComposite"] = (df_model["JobSatisfaction"] +
                                      df_model["EnvironmentSatisfaction"] +
                                      df_model["WorkLifeBalance"]) / 3

drop_cols = [c for c in ["Attrition", "Attrition_Binary", "EmployeeNumber",
                          "OverTime_Binary"] if c in df_model.columns]
X = df_model.drop(columns=drop_cols)
y = df_model["Attrition_Binary"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("Shape:", X.shape)
print("Class balance:", y.value_counts().to_dict())

Shape: (1470, 33)
Class balance: {0: 1233, 1: 237}


In [4]:
print("=== STAGE 1A: BORUTA ===")

rf_boruta = RandomForestClassifier(
    n_estimators=100, class_weight="balanced",
    random_state=42, n_jobs=-1
)
boruta = BorutaPy(
    estimator=rf_boruta, n_estimators="auto",
    perc=90, max_iter=50, random_state=42, verbose=0
)
boruta.fit(X.values, y.values)

strong   = X.columns[boruta.support_].tolist()
tentative = X.columns[boruta.support_weak_].tolist()
boruta_features = strong + tentative

print(f"Original: {X.shape[1]}  →  Boruta kept: {len(boruta_features)}")
print(f"Strong: {len(strong)}, Tentative: {len(tentative)}")
X_boruta = X[boruta_features]

=== STAGE 1A: BORUTA ===
Original: 33  →  Boruta kept: 7
Strong: 6, Tentative: 1


In [5]:
print("\n=== STAGE 1B: RFECV ON BORUTA SURVIVORS ===")

rf_rfe = RandomForestClassifier(
    n_estimators=100, class_weight="balanced",
    random_state=42, n_jobs=-1
)
rfecv = RFECV(
    estimator=rf_rfe,
    step=1,
    cv=skf,
    scoring="roc_auc",
    min_features_to_select=5,
    n_jobs=-1
)
rfecv.fit(X_boruta, y)

final_features = X_boruta.columns[rfecv.support_].tolist()
X_final = X_boruta[final_features]

print(f"Boruta survivors: {X_boruta.shape[1]}  →  RFECV kept: {len(final_features)}")
print(f"Final features: {final_features}")

pd.Series(final_features).to_csv("../outputs/final_features.csv", index=False)

# Plot RFECV scores
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(rfecv.cv_results_["mean_test_score"]) + 1),
         rfecv.cv_results_["mean_test_score"], marker="o", color="#534AB7")
plt.xlabel("Number of features")
plt.ylabel("CV AUC")
plt.title("RFECV — AUC vs feature count")
plt.axvline(len(final_features), color="#D85A30", linestyle="--",
            label=f"Optimal: {len(final_features)} features")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../output/rfecv_curve.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved rfecv_curve.png")


=== STAGE 1B: RFECV ON BORUTA SURVIVORS ===
Boruta survivors: 7  →  RFECV kept: 7
Final features: ['Age', 'MonthlyIncome', 'OverTime', 'TotalWorkingYears', 'SalaryToRoleRatio', 'SatisfactionComposite', 'DailyRate']
Saved rfecv_curve.png


In [6]:
print("\n=== STAGE 2: OPTUNA TUNING ===")

# --- LightGBM ---
def obj_lgbm(trial):
    p = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 500),
        "max_depth":         trial.suggest_int("max_depth", 3, 10),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 20, 100),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "class_weight": "balanced", "random_state": 42, "n_jobs": -1, "verbose": -1
    }
    return cross_val_score(LGBMClassifier(**p), X_final, y,
                           cv=skf, scoring="roc_auc").mean()

study_lgbm = optuna.create_study(direction="maximize")
study_lgbm.optimize(obj_lgbm, n_trials=50, show_progress_bar=True)
print(f"LightGBM best AUC: {study_lgbm.best_value:.4f}")

# --- Gradient Boosting ---
def obj_gb(trial):
    p = {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 300),
        "max_depth":        trial.suggest_int("max_depth", 2, 6),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "random_state": 42
    }
    return cross_val_score(GradientBoostingClassifier(**p), X_final, y,
                           cv=skf, scoring="roc_auc").mean()

study_gb = optuna.create_study(direction="maximize")
study_gb.optimize(obj_gb, n_trials=30, show_progress_bar=True)
print(f"GradientBoosting best AUC: {study_gb.best_value:.4f}")

# --- Random Forest ---
def obj_rf(trial):
    p = {
        "n_estimators":    trial.suggest_int("n_estimators", 100, 400),
        "max_depth":       trial.suggest_int("max_depth", 4, 20),
        "min_samples_leaf":trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features":    trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "class_weight": "balanced", "random_state": 42, "n_jobs": -1
    }
    return cross_val_score(RandomForestClassifier(**p), X_final, y,
                           cv=skf, scoring="roc_auc").mean()

study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(obj_rf, n_trials=30, show_progress_bar=True)
print(f"RandomForest best AUC: {study_rf.best_value:.4f}")


=== STAGE 2: OPTUNA TUNING ===


  0%|          | 0/50 [00:00<?, ?it/s]

LightGBM best AUC: 0.7680


  0%|          | 0/30 [00:00<?, ?it/s]

GradientBoosting best AUC: 0.7741


  0%|          | 0/30 [00:00<?, ?it/s]

RandomForest best AUC: 0.7692


In [8]:
print("\n=== STAGE 3: TABNET ===")

from sklearn.model_selection import train_test_split

X_arr = X_final.values.astype(np.float32)
y_arr = y.values

X_tr, X_val, y_tr, y_val = train_test_split(
    X_arr, y_arr, test_size=0.2, stratify=y_arr, random_state=42
)

tabnet = TabNetClassifier(
    n_d=16, n_a=16, n_steps=4,
    gamma=1.3, n_independent=2, n_shared=2,
    lambda_sparse=1e-3,
    mask_type="sparsemax",
    verbose=0, seed=42
)
tabnet.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric=["auc"],
    max_epochs=100, patience=15,
    batch_size=256, virtual_batch_size=128,
    num_workers=0
)

tabnet_proba_full = tabnet.predict_proba(X_arr)[:, 1]
tabnet_auc = roc_auc_score(y_arr, tabnet_proba_full)
print(f"TabNet AUC: {tabnet_auc:.4f}")

# TabNet built-in feature importance
tabnet_imp = pd.Series(
    tabnet.feature_importances_,
    index=X_final.columns
).sort_values(ascending=False)
tabnet_imp.to_csv("../output/tabnet_importance.csv")
print("Top 5 TabNet features:")
print(tabnet_imp.head(5).round(4))


=== STAGE 3: TABNET ===

Early stopping occurred at epoch 16 with best_epoch = 1 and best_val_0_auc = 0.58644
TabNet AUC: 0.6004
Top 5 TabNet features:
TotalWorkingYears        0.2774
MonthlyIncome            0.2133
Age                      0.1964
DailyRate                0.1906
SatisfactionComposite    0.0792
dtype: float64


In [9]:
print("\n=== STAGE 4: OOF BLENDING ===")

best_lgbm = LGBMClassifier(**study_lgbm.best_params, class_weight="balanced",
                            random_state=42, n_jobs=-1, verbose=-1)
best_gb   = GradientBoostingClassifier(**study_gb.best_params, random_state=42)
best_rf   = RandomForestClassifier(**study_rf.best_params, class_weight="balanced",
                                    random_state=42, n_jobs=-1)

X_np = X_final.values
y_np = y.values

# Generate OOF predictions for each model
def get_oof_predictions(model, X, y, cv):
    oof = np.zeros(len(y))
    for train_idx, val_idx in cv.split(X, y):
        model.fit(X[train_idx], y[train_idx])
        oof[val_idx] = model.predict_proba(X[val_idx])[:, 1]
    return oof

print("Generating OOF predictions...")
oof_lgbm    = get_oof_predictions(best_lgbm, X_np, y_np, skf)
oof_gb      = get_oof_predictions(best_gb,   X_np, y_np, skf)
oof_rf      = get_oof_predictions(best_rf,   X_np, y_np, skf)
oof_tabnet  = tabnet_proba_full   # TabNet already trained on full data

print(f"OOF AUCs — LGBM: {roc_auc_score(y_np, oof_lgbm):.4f} | "
      f"GB: {roc_auc_score(y_np, oof_gb):.4f} | "
      f"RF: {roc_auc_score(y_np, oof_rf):.4f} | "
      f"TabNet: {roc_auc_score(y_np, oof_tabnet):.4f}")

# Optuna finds optimal blend weights
def obj_blend(trial):
    w1 = trial.suggest_float("w_lgbm",   0.0, 1.0)
    w2 = trial.suggest_float("w_gb",     0.0, 1.0)
    w3 = trial.suggest_float("w_rf",     0.0, 1.0)
    w4 = trial.suggest_float("w_tabnet", 0.0, 1.0)
    total = w1 + w2 + w3 + w4
    if total == 0:
        return 0.0
    blended = (w1*oof_lgbm + w2*oof_gb + w3*oof_rf + w4*oof_tabnet) / total
    return roc_auc_score(y_np, blended)

print("Optimizing blend weights with Optuna...")
study_blend = optuna.create_study(direction="maximize")
study_blend.optimize(obj_blend, n_trials=200, show_progress_bar=True)

bw = study_blend.best_params
total_w = bw["w_lgbm"] + bw["w_gb"] + bw["w_rf"] + bw["w_tabnet"]

print(f"\nOptimal blend weights:")
print(f"  LightGBM:  {bw['w_lgbm']/total_w:.3f}")
print(f"  GradBoost: {bw['w_gb']/total_w:.3f}")
print(f"  RandomFor: {bw['w_rf']/total_w:.3f}")
print(f"  TabNet:    {bw['w_tabnet']/total_w:.3f}")
print(f"Blended OOF AUC: {study_blend.best_value:.4f}")

# Final blended predictions on full data
# Refit all models on full data first
best_lgbm.fit(X_np, y_np)
best_gb.fit(X_np, y_np)
best_rf.fit(X_np, y_np)

full_lgbm   = best_lgbm.predict_proba(X_np)[:, 1]
full_gb     = best_gb.predict_proba(X_np)[:, 1]
full_rf     = best_rf.predict_proba(X_np)[:, 1]

blended_proba = (
    bw["w_lgbm"]   * full_lgbm +
    bw["w_gb"]     * full_gb +
    bw["w_rf"]     * full_rf +
    bw["w_tabnet"] * tabnet_proba_full
) / total_w

print(f"Final blended AUC (full data): {roc_auc_score(y_np, blended_proba):.4f}")


=== STAGE 4: OOF BLENDING ===
Generating OOF predictions...
OOF AUCs — LGBM: 0.7675 | GB: 0.7722 | RF: 0.7679 | TabNet: 0.6004
Optimizing blend weights with Optuna...


  0%|          | 0/200 [00:00<?, ?it/s]


Optimal blend weights:
  LightGBM:  0.186
  GradBoost: 0.696
  RandomFor: 0.103
  TabNet:    0.015
Blended OOF AUC: 0.7744
Final blended AUC (full data): 0.8625


In [10]:
print("\n=== STAGE 5: ISOTONIC CALIBRATION ===")

# Calibrate the best individual model (LightGBM)
calibrated = CalibratedClassifierCV(
    estimator=LGBMClassifier(**study_lgbm.best_params, class_weight="balanced",
                              random_state=42, n_jobs=-1, verbose=-1),
    method="isotonic", cv=5
)
calibrated.fit(X_final, y)
cal_proba = calibrated.predict_proba(X_final)[:, 1]

brier_raw = brier_score_loss(y, blended_proba)
brier_cal = brier_score_loss(y, cal_proba)
print(f"Brier score — blended:    {brier_raw:.4f}")
print(f"Brier score — calibrated: {brier_cal:.4f}")
print(f"Improvement: {((brier_raw - brier_cal)/brier_raw * 100):.1f}%")

# Reliability diagram
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, proba, title in [
    (axes[0], blended_proba, "Before calibration (blended)"),
    (axes[1], cal_proba,     "After isotonic calibration")
]:
    prob_true, prob_pred = calibration_curve(y, proba, n_bins=10)
    ax.plot(prob_pred, prob_true, "s-", color="#534AB7", label="Model")
    ax.plot([0,1], [0,1], "--", color="gray", label="Perfect")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.set_title(title)
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../output/calibration_curve.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved calibration_curve.png")


=== STAGE 5: ISOTONIC CALIBRATION ===
Brier score — blended:    0.1020
Brier score — calibrated: 0.0925
Improvement: 9.3%
Saved calibration_curve.png


In [11]:
print("\n=== STAGE 6: THRESHOLD OPTIMIZATION ===")

COST_FN = 0.50   # missed attritor — 50% annual salary
COST_FP = 0.02   # false alarm — 2% salary (retention meeting cost)
avg_annual = df["MonthlyIncome"].mean() * 12

thresholds = np.arange(0.05, 0.95, 0.01)
records = []

for t in thresholds:
    y_pred_t = (cal_proba >= t).astype(int)
    fn = ((y_pred_t == 0) & (y == 1)).sum()
    fp = ((y_pred_t == 1) & (y == 0)).sum()
    tp = ((y_pred_t == 1) & (y == 1)).sum()
    cost = fn * avg_annual * COST_FN + fp * avg_annual * COST_FP
    records.append({
        "threshold": round(t, 2),
        "cost": round(cost, 0),
        "f1": round(f1_score(y, y_pred_t, zero_division=0), 4),
        "tp": int(tp), "fn": int(fn), "fp": int(fp)
    })

thresh_df = pd.DataFrame(records)
best_cost = thresh_df.loc[thresh_df["cost"].idxmin()]
best_f1   = thresh_df.loc[thresh_df["f1"].idxmax()]
default   = thresh_df[thresh_df["threshold"] == 0.50].iloc[0]

print(f"Default  (0.50): cost=${default['cost']:,.0f}  f1={default['f1']:.4f}")
print(f"Optimal (cost):  threshold={best_cost['threshold']}  cost=${best_cost['cost']:,.0f}  f1={best_cost['f1']:.4f}")
print(f"Optimal (F1):    threshold={best_f1['threshold']}  f1={best_f1['f1']:.4f}")
print(f"\nCost saving vs default: ${default['cost'] - best_cost['cost']:,.0f}")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
ax1.plot(thresh_df["threshold"], thresh_df["cost"], color="#D85A30", lw=2)
ax1.axvline(best_cost["threshold"], color="#D85A30", ls="--",
            label=f"Optimal: {best_cost['threshold']}")
ax1.axvline(0.5, color="gray", ls=":", label="Default: 0.5")
ax1.set_ylabel("Business cost ($)"); ax1.set_title("Cost vs threshold")
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(thresh_df["threshold"], thresh_df["f1"], color="#534AB7", lw=2)
ax2.axvline(best_f1["threshold"], color="#534AB7", ls="--",
            label=f"Optimal: {best_f1['threshold']}")
ax2.axvline(0.5, color="gray", ls=":", label="Default: 0.5")
ax2.set_xlabel("Threshold"); ax2.set_ylabel("F1 score")
ax2.set_title("F1 vs threshold"); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../output/threshold_optimization.png", dpi=150, bbox_inches="tight")
plt.close()
thresh_df.to_csv("../output/threshold_analysis.csv", index=False)

OPTIMAL_THRESHOLD = best_cost["threshold"]
print(f"\nFinal threshold: {OPTIMAL_THRESHOLD}")


=== STAGE 6: THRESHOLD OPTIMIZATION ===
Default  (0.50): cost=$6,698,539  f1=0.4125
Optimal (cost):  threshold=0.1  cost=$1,117,464  f1=0.4377
Optimal (F1):    threshold=0.2  f1=0.5841

Cost saving vs default: $5,581,075

Final threshold: 0.1


In [12]:
print("\n=== STAGE 7: SHAP ANALYSIS ===")

explainer  = shap.TreeExplainer(best_lgbm)
shap_vals  = explainer.shap_values(X_final)

if isinstance(shap_vals, list):
    shap_left = shap_vals[1]
elif len(np.array(shap_vals).shape) == 3:
    shap_left = np.array(shap_vals)[:, :, 1]
else:
    shap_left = shap_vals

# Bar
plt.figure()
shap.summary_plot(shap_left, X_final, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("../output/shap_bar.png", dpi=150, bbox_inches="tight")
plt.close()

# Beeswarm
plt.figure()
shap.summary_plot(shap_left, X_final, show=False)
plt.tight_layout()
plt.savefig("../output/shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.close()

# Dependence — OverTime x MonthlyIncome
ot_col = "OverTime_Binary" if "OverTime_Binary" in X_final.columns else "OverTime"
if ot_col in X_final.columns:
    plt.figure()
    shap.dependence_plot(ot_col, shap_left, X_final,
                         interaction_index="MonthlyIncome", show=False)
    plt.tight_layout()
    plt.savefig("../output/shap_dependence.png", dpi=150, bbox_inches="tight")
    plt.close()

# Waterfall for highest-risk employee
high_risk_idx = np.argmax(cal_proba)
base_val = explainer.expected_value
if isinstance(base_val, (list, np.ndarray)):
    base_val = float(np.array(base_val).flat[-1])

shap_exp = shap.Explanation(
    values=shap_left[high_risk_idx],
    base_values=base_val,
    data=X_final.iloc[high_risk_idx].values,
    feature_names=X_final.columns.tolist()
)
plt.figure()
shap.waterfall_plot(shap_exp, show=False)
plt.tight_layout()
plt.savefig("../output/shap_waterfall.png", dpi=150, bbox_inches="tight")
plt.close()
print("All SHAP plots saved.")


=== STAGE 7: SHAP ANALYSIS ===
All SHAP plots saved.


<Figure size 640x480 with 0 Axes>

In [13]:
print("\n" + "="*55)
print("FINAL PIPELINE SUMMARY")
print("="*55)

print(f"\nFeature selection:  {X.shape[1]} → {len(final_features)} features")
print(f"  Boruta removed:   {X.shape[1] - len(boruta_features)}")
print(f"  RFECV removed:    {len(boruta_features) - len(final_features)}")

print(f"\nModel AUCs (OOF):")
print(f"  LightGBM:         {roc_auc_score(y_np, oof_lgbm):.4f}")
print(f"  GradientBoosting: {roc_auc_score(y_np, oof_gb):.4f}")
print(f"  RandomForest:     {roc_auc_score(y_np, oof_rf):.4f}")
print(f"  TabNet:           {tabnet_auc:.4f}")
print(f"  OOF Blend:        {study_blend.best_value:.4f}")

print(f"\nCalibration (Brier score):")
print(f"  Before: {brier_raw:.4f}  After: {brier_cal:.4f}")

print(f"\nThreshold optimization:")
print(f"  Default (0.5) cost: ${default['cost']:,.0f}")
print(f"  Optimal threshold:  {OPTIMAL_THRESHOLD}")
print(f"  Optimized cost:     ${best_cost['cost']:,.0f}")
print(f"  Saving:             ${default['cost']-best_cost['cost']:,.0f}")

joblib.dump(calibrated,  "../output/final_model.pkl")
joblib.dump(best_lgbm,   "../output/lgbm_model.pkl")
print(f"\nModels saved.")

print("\nOutput files:")
for f in sorted(os.listdir("../output")):
    print(f"  {f}")

print("\nPipeline complete!")


FINAL PIPELINE SUMMARY

Feature selection:  33 → 7 features
  Boruta removed:   26
  RFECV removed:    0

Model AUCs (OOF):
  LightGBM:         0.7675
  GradientBoosting: 0.7722
  RandomForest:     0.7679
  TabNet:           0.6004
  OOF Blend:        0.7744

Calibration (Brier score):
  Before: 0.1020  After: 0.0925

Threshold optimization:
  Default (0.5) cost: $6,698,539
  Optimal threshold:  0.1
  Optimized cost:     $1,117,464
  Saving:             $5,581,075

Models saved.

Output files:
  best_rf_model.pkl
  calibration_curve.png
  cost_of_attrition.csv
  cost_of_attrition0.csv
  feature_names.csv
  final_model.pkl
  lgbm_model.pkl
  model_comparison.csv
  rfecv_curve.png
  shap_bar.png
  shap_beeswarm.png
  shap_dependence.png
  shap_importance.png
  shap_waterfall.png
  stacking_model.pkl
  tabnet_importance.csv
  threshold_analysis.csv
  threshold_optimization.png
  xgb_booster.json

Pipeline complete!
